# Multidimensional Data, Distance Functions, and Matrices

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
np.set_printoptions(legacy='1.25')

In [ ]:
countiesDF = pd.read_csv("Counties2010.csv")
countiesDF

#### Scatter Plot Matrix

In [ ]:
pd.plotting.scatter_matrix(countiesDF, figsize=(15,15))

#### Parallel Coordinates Plot

Let's group our data first to reduce our number of observations (PC plots are already really messy)

In [ ]:
# Split the dataset into groups by state, creating a GroupBy object
g = countiesDF.groupby("State")
g

In [ ]:
len(g)

In [ ]:
# What is the size of each group
g.size()

In [ ]:
# Get the sum of the population column for each group (i.e., the population of the state)
g.Population.sum()

In [ ]:
# Get the means for all the columns for all the groups (states)
g.mean(numeric_only=True)

In [ ]:
# Assign the means by group to a DataFrame object
groupedDF = g.mean(numeric_only=True)
groupedDF["State"] = groupedDF.index
groupedDF

In [ ]:
# Create a (poor) parallel coordinates plot of the grouped data
pc = pd.plotting.parallel_coordinates(groupedDF, "State", colormap=plt.cm.rainbow)
pc.figure.set_size_inches(15, 7)
pc.legend(ncols=5)

In [ ]:
# Load a dataset from the Seaborn library
data = sns.load_dataset('iris')
data

In [ ]:
# Plot the iris data on a parallel coordinates plot
irisPC = pd.plotting.parallel_coordinates(data, 'species', colormap=plt.get_cmap("Set2"), alpha=0.25)
plt.title('Parallel Coordinates Plot of Iris Dataset')
plt.show()

## Normalizing Data

#### Feature Scaling or Min-Max Normalizing

In [ ]:
# Min-Max Normalizing maps everything to a [0-1] scale
# scaled = (n - min) / (max - min)

numericGroupedDF = groupedDF.iloc[:, :-1]
dfScale = (numericGroupedDF - numericGroupedDF.min()) / (numericGroupedDF.max() - numericGroupedDF.min())
dfScale["State"] = dfScale.index
dfScale

In [ ]:
pc = pd.plotting.parallel_coordinates(dfScale, "State", colormap=plt.cm.rainbow)
pc.figure.set_size_inches(15, 10)
pc.legend(ncols=5)

#### Z-Score Normalizing

In [ ]:
# z-score normalization computes the number of standard deviations each point is from the mean in each dimension
# z = (n-mean)/std

dfZ = (numericGroupedDF - numericGroupedDF.mean()) / numericGroupedDF.std()
dfZ["State"] = dfZ.index
dfZ

In [ ]:
pc = pd.plotting.parallel_coordinates(dfZ, "State", colormap=plt.cm.rainbow)
pc.figure.set_size_inches(15, 10)
pc.legend(ncols=5)

## Distance Functions

### Quantitative data
Is VA more like MD, NC, or CA?

In [ ]:
statesDF = pd.read_csv("State_demographics.csv")
statesDF.index = statesDF.Abbrev
numericStatesDF = statesDF.iloc[:, 2:]
numericStatesDF

In [ ]:
# Subset the data to just VA, NC, MD, CA, WA
normStatesDF = (numericStatesDF - numericStatesDF.mean()) / numericStatesDF.std()
normStatesDF.loc[['VA','NC','MD','CA','WA']]

#### L1 Distance (Manhattan, TaxiCab)

$dist(P,Q) = \sum_{i} |P_i-Q_i|$

Let's compare the distance of VA to each of our other selected states using L1 distance

In [ ]:
(normStatesDF.loc["VA"] - normStatesDF.loc["NC"]).abs().sum()

In [ ]:
(normStatesDF.loc["VA"] - normStatesDF.loc["MD"]).abs().sum()

In [ ]:
(normStatesDF.loc["VA"] - normStatesDF.loc["CA"]).abs().sum()

In [ ]:
(normStatesDF.loc["VA"] - normStatesDF.loc["WA"]).abs().sum()

#### L2 Distance (Euclidean)

$dist(P,Q) = \sqrt{\sum_{i} (P_i-Q_i)^{2}}$

Let's compare the distance of VA to each of our other selected states using L2 distance

In [ ]:
((normStatesDF.loc["VA"] - normStatesDF.loc["NC"])**2).sum()**0.5

In [ ]:
((normStatesDF.loc["VA"] - normStatesDF.loc["MD"])**2).sum()**0.5

In [ ]:
((normStatesDF.loc["VA"] - normStatesDF.loc["CA"])**2).sum()**0.5

In [ ]:
((normStatesDF.loc["VA"] - normStatesDF.loc["WA"])**2).sum()**0.5

#### Lp Distance (Minkowski)

In [ ]:
print((((normStatesDF.loc["VA"] - normStatesDF.loc["WA"]).abs())**1).sum()**(1/1))
print((((normStatesDF.loc["VA"] - normStatesDF.loc["WA"]).abs())**2).sum()**(1/2))
print((((normStatesDF.loc["VA"] - normStatesDF.loc["WA"]).abs())**3).sum()**(1/3))
print((((normStatesDF.loc["VA"] - normStatesDF.loc["WA"]).abs())**3.14).sum()**(1/3.14))
print((((normStatesDF.loc["VA"] - normStatesDF.loc["WA"]).abs())**5).sum()**(1/5))
print((((normStatesDF.loc["VA"] - normStatesDF.loc["WA"]).abs())**12).sum()**(1/12))

### Categorical data
Is DC more like Atlanta or NY?

In [ ]:
citiesDF = pd.read_csv("Cities.csv")
citiesDF.index = citiesDF.City
citiesDF = citiesDF.iloc[:, 1:]
citiesDF

In [ ]:
# Subset our data to Washington DC, Algiers, Adelaide, Zurich, Warsaw
citiesDF.loc[["Washington DC", "Algiers", "Adelaide", "Zurich", "Warsaw"]]

#### Hamming distance (String, Edit)

$dist(P,Q) = \sum_{i}
\begin{cases}
    0,& \text{if } P_i == Q_i\\
    1,              & \text{otherwise}
\end{cases}$

Let's compare the distance of Washington, DC to each of our other selected states using Hamming distance

In [ ]:
(citiesDF.loc["Washington DC"] != citiesDF.loc["Algiers"]).sum()

In [ ]:
(citiesDF.loc["Washington DC"] != citiesDF.loc["Adelaide"]).sum()

In [ ]:
(citiesDF.loc["Washington DC"] != citiesDF.loc["Zurich"]).sum()

In [ ]:
(citiesDF.loc["Washington DC"] != citiesDF.loc["Warsaw"]).sum()

In [ ]:
(citiesDF.loc["Zurich"] != citiesDF.loc["Warsaw"]).sum()

## Calculating a Distance Matrix

$M = \begin{bmatrix} 
0 & d_{12} & d_{13} & ... & d_{1n} \\ 
d_{21} & 0 & d_{23} & ... & d_{2n} \\ 
d_{31} & d_{32} & 0 & ... & d_{3n} \\ 
... & ... & ... & ... & ... \\ 
d_{n1} & d_{n2} & d_{n3} & ... & 0 \end{bmatrix}$

$M_{i,j} = d_{ij} = distance(i, j)$


Computing L1 distance matrix via Map (not using For loops).
1. distance between 2 given states?   row -> scalar
2. distances of every state to a given state?  n rows -> vector of length n
3. distances of every state to every state?  n rows -> n x n matrix

In [ ]:
# Compare VA to one state
(normStatesDF.loc["MD"] - normStatesDF.loc["VA"]).abs().sum()

In [ ]:
# Compare VA to every other state
# Look at how column values for each row differ
(normStatesDF - normStatesDF.loc["VA"]).abs().sum(axis=1) # Do the operation on the columns, i.e., differences down a single column

In [ ]:
# Compare every state to every other state
normStatesDF.apply(lambda row: (normStatesDF - row).abs().sum(axis=1), axis=1) # Now we want to apply this operation down the column, i.e., on every one of our states

# Apply lets us apply a function to many items in a DF object, in this case the rows

# Lambda expressions are a concise way to define small, unnamed functions directly within the code where they are used.
# e.g., lambda arguments: expression

## Dimension Reduction

### Multi-Dimensional Scaling (MDS):
1. Select only appropriate variables (e.g., quantitative, categorical)
2. Normalize your data
3. Calculate all pairwise distances
4. Solve for low-dimensional coordinates so that the difference between the measured distances of each pair of points in the low-dimensional space
is as close as possible to the difference between measured distances in the high-dimensional space

In [ ]:
import sklearn

In [ ]:
import sklearn.metrics.pairwise as pairs
import sklearn.manifold as MDScale

In [ ]:
# Step 1: Subset the data
statesDF2 = pd.read_csv("States2010.csv")
statesDF2.index = statesDF2.Name
numericStatesDF2 = statesDF2.iloc[:, 2:]
numericStatesDF2

In [ ]:
# Step 2: Normalize the data
normStatesDF2 = (numericStatesDF2 - numericStatesDF2.mean()) / numericStatesDF2.std()
normStatesDF2

In [ ]:
# Step 3: Calculate pairwise distances
# distHD = pairs.manhattan_distances(normStatesDF2)
distHD = pairs.euclidean_distances(normStatesDF2)
distHD = pd.DataFrame(distHD, index=normStatesDF2.index, columns=normStatesDF2.index) # Turn it back into a DF object
distHD

In [ ]:
# Step 4: Calculate the optimal low-dimensional distances
mds = MDScale.MDS(n_components=2, max_iter=1000, n_init=5, dissimilarity="precomputed") # Reduce to 2 components, 5 runs w/ new starting point
data2D = mds.fit_transform(distHD) # Transform our fitted data to a numpy array
data2D = pd.DataFrame(data2D, columns=['x','y'], index=normStatesDF2.index) # Assign that to a DataFrame object
data2D

In [ ]:
# Plot resulting MDS map with labels
ax = data2D.plot.scatter(x='x', y='y', figsize=(15,15))
for i in range(len(data2D)): # Add data labels
    ax.text(data2D.iloc[i]['x']+0.1, data2D.iloc[i]['y']+0.1, data2D.index[i]) 